# センサーの最適配置（元の課題）: Amplify の QUBO と古典的手法の比較

国際航業「センサーの最適配置」を、次の手法で解いて比べます。式は `placement/README.md` にあります。

| 手法 | 中身 |
|---|---|
| MILP（古典） | HiGHS による厳密解。最適であることの証明付き |
| 貪欲法（古典） | コスト当たりの不足の減り方が最大の手を繰り返す |
| QUBO 1 | 課題をそのまま Amplify の制約（`greater_equal` / `less_equal`）で書いたもの。スラック変数あり |
| QUBO 2 | ラグランジュ緩和（スラック変数なし）。乗数を外側のループで調整 |
| ＋修復 | QUBO の解の不足を貪欲法の手で補い、要らないセンサーを外したもの |

**Amplify のトークンを入れると QUBO を Fixstars Amplify AE で**解きます。空欄なら、同じ QUBO をローカルの焼きなまし（neal）で解きます。

## 1. 準備

In [ ]:
!pip install -q amplify dimod dwave-neal
!apt-get -qq install -y fonts-ipafont-gothic > /dev/null
!rm -rf Q-Questgrow- && git clone -q -b claude/request-5v4fsc https://github.com/mori560269/Q-Questgrow-.git
%cd Q-Questgrow-/placement
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
try:
    fm.fontManager.addfont('/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf')
    plt.rcParams['font.family'] = ['IPAGothic', 'DejaVu Sans']
except Exception:
    pass
import sys, json, numpy as np, pandas as pd
sys.path.insert(0, '.')

## 2. 設定（ここだけ変えれば条件を変えられます）

In [ ]:
from getpass import getpass
token = getpass('Amplify のアクセストークン (空のまま Enter でローカルの焼きなまし) ') or None
backend = 'amplify' if token else 'neal'

SEEDS = 3            # QUBO を何回解くか (乱数の種)
Q1_TIME_MS = 10000   # QUBO 1: AE の制限時間 [ms] / 1 回
Q1_SWEEPS = 8000     # QUBO 1: neal のスイープ数
Q2_ITERS = 100       # QUBO 2: 乗数の外側ループの回数
Q2_TIME_MS = 500     # QUBO 2: AE の制限時間 [ms] / 内側 1 回
Q2_SWEEPS = 200      # QUBO 2: neal のスイープ数 / 内側 1 回
print('QUBO の解き方:', 'Fixstars Amplify AE' if backend == 'amplify' else 'ローカルの焼きなまし (neal)')

## 3. 全手法の比較（コスト・時間・実行可能か）

In [ ]:
from experiments import run_all
prob, rows = run_all(backend, token, seeds=SEEDS, q1_sweeps=Q1_SWEEPS, q1_time_ms=Q1_TIME_MS,
                     q2_iters=Q2_ITERS, q2_sweeps=Q2_SWEEPS, q2_time_ms=Q2_TIME_MS)
table = pd.DataFrame([{k: v for k, v in r.items() if k != '_x'} for r in rows])
table

配置の地図（実行可能な解だけ）

In [ ]:
from common import save_map
from IPython.display import Image
maps = {}
for r in rows:
    if r.get('_x') is not None and r['実行可能']:
        maps[r['手法']] = prob.evaluate(np.array(r['_x']), cell_m=1000)
save_map(prob, maps, 'placement_maps.png')
Image('placement_maps.png')

## 4. 同じ時間でどこまで良い解が出るか

QUBO 2 は、反復回数を時間の予算に合わせて変えます。MILP は時間制限を変えます。
時間切れのときは、それまでに見つけた解を使います。

In [ ]:
from experiments import equal_time
et = pd.DataFrame(equal_time(backend, token, budgets=(1, 3, 10)))
et

## 5. 規模を変えた比較（候補地を増やす）

元の 86 か所に、格子状の候補地を足して問題を大きくします。
- `8000`: 8 km 間隔、184 か所
- `5000`: 5 km 間隔、342 か所
- `3500`: 3.5 km 間隔、611 か所

ローカルの焼きなましでは 611 か所に数分かかります。

In [ ]:
from experiments import scaling
sc = pd.DataFrame(scaling(backend, token, grids=(None, 8000, 5000), q2_iters=Q2_ITERS,
                          q2_sweeps=Q2_SWEEPS, q2_time_ms=Q2_TIME_MS))
sc

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(sc['候補地'], sc['MILP_秒'], 'o-', label='MILP')
ax[0].plot(sc['候補地'], sc['QUBO2修復_秒'], 's-', label='QUBO 2 + 修復')
ax[0].set_xlabel('候補地の数'); ax[0].set_ylabel('時間 [秒]'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(sc['候補地'], sc['MILP_コスト'], 'o-', label='MILP (最適)')
ax[1].plot(sc['候補地'], sc['QUBO2修復_コスト'], 's-', label='QUBO 2 + 修復')
ax[1].set_xlabel('候補地の数'); ax[1].set_ylabel('総調達コスト'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. 結果を保存

In [ ]:
from google.colab import files
out = dict(backend=backend, settings=dict(SEEDS=SEEDS, Q1_TIME_MS=Q1_TIME_MS, Q1_SWEEPS=Q1_SWEEPS,
           Q2_ITERS=Q2_ITERS, Q2_TIME_MS=Q2_TIME_MS, Q2_SWEEPS=Q2_SWEEPS),
           compare=table.to_dict('records'), equal_time=et.to_dict('records'), scaling=sc.to_dict('records'))
json.dump(out, open(f'placement_results_{backend}.json', 'w'), ensure_ascii=False, indent=1, default=str)
files.download(f'placement_results_{backend}.json')
files.download('placement_maps.png')